<a href="https://colab.research.google.com/github/hoofs-lab/Prueba_Peloton/blob/main/Peloton_10k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEC EDGAR to SQLite Pipeline: Peloton 2022

In [16]:
# Step 1: Install necessary dependencies
!pip install -q pymsql edgartools pandas

In [17]:
# Step 2: Load SQL extension for easy database querying
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 1. Google Drive & Directory Setup

In [18]:
# Step 4: Bind Jupysql to your newly populated Google Drive database
%sql sqlite:///{db_path}

In [ ]:
%%sql
--Sql query here

## 2. SEC EDGAR Authentication & Data Download

In [3]:
!pip install -q edgartools pandas

In [19]:
import pandas as pd
from edgar import Company, set_identity

# Identify with SEC (required by EDGAR terms of service)
set_identity("CarlosGuevara hoofsdata@gmail.com")

In [20]:
# Initialize Peloton by ticker and fetch 2022 filings
pton = Company("PTON")
filings_2022 = pton.get_filings(form=["10-K", "10-Q"], filing_date="2022-01-01:2022-12-31")

# Display retrieved filings
print(filings_2022)

Filings: 4 | PELOTON INTERACTIVE, INC. | 2022-02-08 to 2022-11-03
   #  Filed       Form
   0. 2022-11-03  10-Q
   1. 2022-09-07  10-K
   2. 2022-05-10  10-Q
   3. 2022-02-08  10-Q
[filings[n] to select | .filter(form=) to narrow]


In [21]:
# Step 3: Run the database writing and normalization loop
import sqlite3

# Clean run option: Uncomment the next lines to reset the database file entirely on start
# if os.path.exists(db_path):
#     os.remove(db_path)
#     print("Reset existing database file.")

conn = sqlite3.connect(db_path)

for filing in filings_2022:
    print(f"Processing {filing.form} filed on {filing.filing_date}...")

    xbrl = filing.xbrl()
    if not xbrl:
        continue

    statements = xbrl.statements
    target_statements = {
        "income_statement": statements.income_statement(),
        "balance_sheet": statements.balance_sheet(),
        "cash_flow_statement": statements.cash_flow_statement()
    }

    for table_name, stmt in target_statements.items():
        if stmt is not None:
            df = stmt.to_dataframe()
            fixed_cols = ["concept", "label"] if "label" in df.columns else ["concept"]
            date_cols = [col for col in df.columns if col not in fixed_cols]

            # Normalize/melt dynamic period columns into single rows
            df_melted = df.melt(
                id_vars=fixed_cols,
                value_vars=date_cols,
                var_name="period_date_label",
                value_name="value"
            )

            df_melted["form_type"] = filing.form
            df_melted["filing_date"] = str(filing.filing_date)
            df_melted["period_end"] = str(filing.period_of_report)

            df_melted.to_sql(table_name, conn, if_exists="append", index=False)

conn.close()
print(f"Done! All statements normalized and saved successfully to Google Drive: {db_path}")

Processing 10-Q filed on 2022-11-03...
Processing 10-K filed on 2022-09-07...
Processing 10-Q filed on 2022-05-10...
Processing 10-Q filed on 2022-02-08...
Done! All statements normalized and saved successfully to Google Drive: /content/drive/MyDrive/Peloton_Data/peloton_2022.db


In [22]:
# Step 5: Verify the written tables and run a quick test query
conn = sqlite3.connect(db_path)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables inside Google Drive database:\n", tables)

# Query preview
df_preview = pd.read_sql("SELECT * FROM income_statement LIMIT 5;", conn)
display(df_preview)

conn.close()

Tables inside Google Drive database:
                   name
0     income_statement
1        balance_sheet
2  cash_flow_statement


,concept,label,period_date_label,value,form_type,filing_date,period_end
0,us-gaap_RevenueFromContractWithCustomerExcludi...,Total revenue,standard_concept,Revenue,10-Q,2022-11-03,2022-09-30
1,us-gaap_RevenueFromContractWithCustomerExcludi...,Connected Fitness Products,standard_concept,None,10-Q,2022-11-03,2022-09-30
2,us-gaap_RevenueFromContractWithCustomerExcludi...,Subscription,standard_concept,None,10-Q,2022-11-03,2022-09-30
3,us-gaap_RevenueFromContractWithCustomerExcludi...,North America,standard_concept,None,10-Q,2022-11-03,2022-09-30
4,us-gaap_RevenueFromContractWithCustomerExcludi...,International,standard_concept,None,10-Q,2022-11-03,2022-09-30


In [23]:
from google.colab import drive
import os
import sqlite3

# Mount Google Drive
drive.mount('/content/drive')

# Set up target directory for the database
drive_folder_path = "/content/drive/MyDrive/Peloton_Data"
if not os.path.exists(drive_folder_path):
    os.makedirs(drive_folder_path)
    print(f"Created new folder at: {drive_folder_path}")
else:
    print(f"Folder already exists at: {drive_folder_path}")

db_path = os.path.join(drive_folder_path, "peloton_2022.db")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder already exists at: /content/drive/MyDrive/Peloton_Data


In [14]:
import os
import sqlite3

# Define the target folder path in your Google Drive
drive_folder_path = "/content/drive/MyDrive/Peloton_Data"

# Create the folder if it does not exist already
if not os.path.exists(drive_folder_path):
    os.makedirs(drive_folder_path)
    print(f"Created new folder at: {drive_folder_path}")
else:
    print(f"Folder already exists at: {drive_folder_path}")

# Define the full path to your database within this folder
db_path = os.path.join(drive_folder_path, "peloton_2022.db")

# Connect (or create) the database in your specific folder
conn = sqlite3.connect(db_path)
print(f"Connected to database at: {db_path}")

Created new folder at: /content/drive/MyDrive/Peloton_Data
Connected to database at: /content/drive/MyDrive/Peloton_Data/peloton_2022.db
